# Black Sigatoka preprocessing

This notebook prepares the source-time-separated synthetic scenarios for a first regression baseline. It does not train a model. The target is a research-informed synthetic score, not a confirmed disease observation. Run it from the project root with the `.venv` Jupyter kernel.

## 1. Set up the project and protect source files

We load the small reusable module from `src/`. Before reading the approved partitions, record hashes for the CSV files under the Black Sigatoka dataset folder. The final cell checks these files were not changed.

In [ ]:
from pathlib import Path 
import hashlib
import json
import sys
import platform

import numpy as np
import pandas as pd
import sklearn
import xgboost

ROOT = Path.cwd().resolve()
while not (ROOT / 'datasets' / 'black_sigatoka' / 'revised_splits').is_dir():
    if ROOT.parent == ROOT:
        raise FileNotFoundError('Could not find the project datasets folder from the current directory.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from black_sigatoka_preprocessing import (
    FEATURES, TARGET, CATEGORY_BOUNDS, EXPECTED_ROWS,
    load_and_validate_datasets, prepare_model_inputs, verify_source_time_separation,
    validate_prediction_input, risk_score_to_category,
)

DATA_DIR = ROOT / 'datasets' / 'black_sigatoka'
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

source_csvs = sorted(DATA_DIR.rglob('*.csv'))
source_hashes_before = {str(path.relative_to(ROOT)): sha256(path) for path in source_csvs}
print(f'Project root: {ROOT}')
print(f'CSV files protected by SHA-256 check: {len(source_hashes_before)}')

Project root: C:\Users\HP\Desktop\ML training
CSV files protected by SHA-256 check: 9


## 2. Load and validate the approved partitions

The module checks the documented row counts, exact shared schema, required columns, missing and non-finite selected values, score range, wetness inputs, category labels, and agreement with the model input contract. It raises a clear error instead of repairing or dropping data.

In [2]:
frames, validation_summaries = load_and_validate_datasets(ROOT)
# Re-read donor timestamps from the actual revised CSVs and recalculate pairwise gaps.
source_time_results = verify_source_time_separation(ROOT)
display(source_time_results.style.format({'minimum_gap_hours': '{:,.0f}'}))
separation_path = ROOT / 'reports' / 'black_sigatoka' / 'final_safeguards' / 'source_time_separation.csv'
if not separation_path.exists():
    raise FileNotFoundError(f'Required source-time safeguard report is missing: {separation_path}')
expected_pairs = {'train-validation', 'train-test', 'validation-test'}
saved_separation = pd.read_csv(separation_path).set_index('pair').sort_index()
fresh_separation = source_time_results.set_index('pair').sort_index()
if set(fresh_separation.index) != expected_pairs or set(saved_separation.index) != expected_pairs:
    raise ValueError('Fresh or saved source-time report does not contain the three expected split pairs.')
integer_columns = ['left_donor_count', 'right_donor_count', 'shared_donor_timestamps']
for column in integer_columns:
    if not np.array_equal(saved_separation[column].to_numpy(dtype='int64'), fresh_separation[column].to_numpy(dtype='int64')):
        raise ValueError(f'Saved source-time report disagrees with the CSV timestamps for {column}.')
if not np.allclose(saved_separation['minimum_gap_hours'].to_numpy(dtype=float), fresh_separation['minimum_gap_hours'].to_numpy(dtype=float), rtol=0, atol=1e-9):
    raise ValueError('Saved source-time report disagrees with the freshly calculated minimum time gaps.')
if not np.array_equal(saved_separation['passes_120_hour_gap'].astype(bool).to_numpy(), fresh_separation['passes_120_hour_gap'].astype(bool).to_numpy()):
    raise ValueError('Saved source-time report disagrees with the freshly calculated 120-hour checks.')
print('Fresh timestamp calculations agree with the existing safeguard report.')
dataset_validation_summary = pd.DataFrame(validation_summaries.values())
display(dataset_validation_summary)
assert {name: len(frame) for name, frame in frames.items()} == EXPECTED_ROWS

,pair,left_donor_count,right_donor_count,shared_donor_timestamps,minimum_gap_hours,passes_120_hour_gap
0,train-validation,2583,699,0,265,True
1,train-test,2583,484,0,"4,657",True
2,validation-test,699,484,0,267,True


Fresh timestamp calculations agree with the existing safeguard report.


,split,source_file,rows,columns,schema_consistent,selected_missing_values,selected_nonfinite_values,risk_score_min,risk_score_max,risk_category_mismatches,input_shape,target_shape
0,train,datasets/black_sigatoka/revised_splits/BS_SOUR...,4501,25,True,0,0,0.0,99.9950,0,4501 x 3,4501
1,validation,datasets/black_sigatoka/revised_splits/BS_SOUR...,937,25,True,0,0,0.0,99.9939,0,937 x 3,937
2,test,datasets/black_sigatoka/revised_splits/BS_SOUR...,524,25,True,0,0,0.0,99.9946,0,524 x 3,524


## 3. Select the model inputs and targets

The feature contract allows exactly three inputs in a fixed order. We keep the source-time partitions intact. No merge, shuffle, resampling, SMOTE, or row removal is performed. The test partition remains reserved for the later final evaluation.

In [3]:
X_by_split, y_by_split = prepare_model_inputs(frames)
X_train, X_val, X_test = (X_by_split[name] for name in ('train', 'validation', 'test'))
y_train, y_val, y_test = (y_by_split[name] for name in ('train', 'validation', 'test'))

for split, X, y in [('train', X_train, y_train), ('validation', X_val, y_val), ('test', X_test, y_test)]:
    assert X.columns.tolist() == FEATURES
    assert len(X) == len(y)
    assert X.shape[1] == 3
    assert np.isfinite(X.to_numpy(dtype=float)).all()
    assert np.isfinite(y.to_numpy(dtype=float)).all()
    print(f'{split}: X={X.shape}, y={y.shape}')

for forbidden in [TARGET, 'risk_level', 'scenario_id', 'split', 'weather_analogue_timestamp_utc']:
    assert forbidden not in X_train.columns
print('Approved feature order:', FEATURES)
print('Target:', TARGET)
print('Selected missing values:', {name: int(X.isna().sum().sum()) for name, X in X_by_split.items()})
print('Scaling: not applied (tree-based models do not require conventional scaling).')
print('Imputation: not applied (selected inputs contain no missing values).')

train: X=(4501, 3), y=(4501,)
validation: X=(937, 3), y=(937,)
test: X=(524, 3), y=(524,)
Approved feature order: ['air_temp_c', 'estimated_leaf_wet', 'consecutive_wetness_hours']
Target: risk_score
Selected missing values: {'train': 0, 'validation': 0, 'test': 0}
Scaling: not applied (tree-based models do not require conventional scaling).
Imputation: not applied (selected inputs contain no missing values).


## 4. Validate prediction inputs and risk categories

The helper accepts a mapping or a DataFrame containing only the approved features. Temperature and the current wetness state may come from current readings. Consecutive wetness hours require preceding hourly wetness history; one instantaneous sensor reading cannot determine this duration. No future observations are used.

Category boundaries follow the documented score rules exactly. Because `No Risk` requires an exact zero, a small nonzero regression prediction falls in `Low`; this boundary deserves attention during later evaluation. Keep raw model predictions available there, and report any predictions outside 0–100 before considering bounded values for inference.

In [4]:
boundary_checks = {0: 'No Risk', 0.001: 'Low', 24.999: 'Low', 25: 'Moderate', 74.999: 'Moderate', 75: 'High', 100: 'High'}
for score, expected in boundary_checks.items():
    assert risk_score_to_category(score) == expected

valid_example = validate_prediction_input({
    'air_temp_c': 27.2,
    'estimated_leaf_wet': 1,
    'consecutive_wetness_hours': 12,
})
assert valid_example.columns.tolist() == FEATURES and valid_example.shape == (1, 3)

def must_fail(call, expected_exception):
    try:
        call()
    except expected_exception:
        return
    raise AssertionError(f'Expected {expected_exception.__name__} was not raised.')

must_fail(lambda: validate_prediction_input({'air_temp_c': 27, 'estimated_leaf_wet': 2, 'consecutive_wetness_hours': 4}), ValueError)
must_fail(lambda: validate_prediction_input({'air_temp_c': float('nan'), 'estimated_leaf_wet': 1, 'consecutive_wetness_hours': 4}), ValueError)
must_fail(lambda: validate_prediction_input({'air_temp_c': 27, 'estimated_leaf_wet': 1, 'consecutive_wetness_hours': -1}), ValueError)
must_fail(lambda: risk_score_to_category(100.01), ValueError)
must_fail(lambda: risk_score_to_category(float('inf')), ValueError)
must_fail(lambda: risk_score_to_category('25'), TypeError)
print('Valid prediction input and all category boundary/error examples passed.')

Valid prediction input and all category boundary/error examples passed.


## 5. Save the preprocessing record

The report and configuration describe the processed splits, software versions, decisions, and scientific limits. The separate NASA historical dataset remains outside model training and will be considered in a later evaluation stage. Its environmental distribution differs from these simulated scenarios, and its labels are proxy scores rather than confirmed infections. No duplicate data files or model artifacts are saved.

In [5]:
preprocessing_dir = ROOT / 'reports' / 'black_sigatoka' / 'preprocessing'
preprocessing_dir.mkdir(parents=True, exist_ok=True)

config = {
    'purpose': 'Preprocessing for preliminary synthetic risk-score regression; no model training.',
    'features': FEATURES,
    'target': TARGET,
    'risk_category_bounds': CATEGORY_BOUNDS,
    'source_datasets': {name: summary['source_file'] for name, summary in validation_summaries.items()},
    'row_counts': {name: summary['rows'] for name, summary in validation_summaries.items()},
    'python_version': platform.python_version(),
    'library_versions': {'pandas': pd.__version__, 'numpy': np.__version__, 'scikit_learn': sklearn.__version__, 'xgboost': xgboost.__version__},
    'imputation_applied': False,
    'scaling_applied': False,
    'synthetic_label_limitations': [
        'Risk scores are research-informed synthetic mathematical labels, not confirmed disease observations.',
        'The upstream continuous score generator is unavailable; six repeated three-feature combinations have small score differences.',
        'A successful model measures reproduction of the synthetic scoring relationship, not field disease prediction.'
    ],
    'source_csv_sha256_before': source_hashes_before,
}
(preprocessing_dir / 'preprocessing_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
dataset_validation_summary.to_csv(preprocessing_dir / 'dataset_validation_summary.csv', index=False)

report = f'''# Black Sigatoka preprocessing report

## Scope

Prepared the revised source-time-separated partitions for a preliminary regression baseline. Temporal separation was independently reverified from the actual source CSV timestamps and compared with the saved safeguard report. No model was trained, and no source dataset was edited. The labels are synthetic environmental risk scores, not confirmed Black Sigatoka outcomes.

## Inputs and validation

| Split | Rows | Columns | X shape | y shape | Missing selected values | Non-finite selected values | Category mismatches |
|---|---:|---:|---|---|---:|---:|---:|
{chr(10).join(f"| {r['split']} | {r['rows']} | {r['columns']} | {r['input_shape']} | {r['target_shape']} | {r['selected_missing_values']} | {r['selected_nonfinite_values']} | {r['risk_category_mismatches']} |" for r in validation_summaries.values())}

Feature order: `{', '.join(FEATURES)}`. Target: `{TARGET}`. The model input contract was checked against this specification. The source-time partitions were preserved; no shuffling, resampling, class balancing, SMOTE, merge, or row removal was applied.

### Independent donor-time verification

Timestamps were reparsed as UTC directly from each revised CSV. The calculation found no missing or invalid timestamps, no cross-split timestamp overlaps, and minimum gaps greater than 120 hours. Fresh results were compared with the existing safeguard CSV and matched.

| Split pair | Unique timestamps in first split | Unique timestamps in second split | Shared timestamps | Minimum gap (hours) | Gap > 120 hours |
|---|---:|---:|---:|---:|---|
{chr(10).join(f"| {r['pair']} | {r['left_donor_count']} | {r['right_donor_count']} | {r['shared_donor_timestamps']} | {r['minimum_gap_hours']:,.0f} | {r['passes_120_hour_gap']} |" for r in source_time_results.to_dict(orient='records'))}

## Preprocessing decisions

The approved features contain no missing values, so no imputation or row removal was needed. Features were not scaled because the planned Random Forest and XGBoost regressors are tree-based. Inputs and targets were verified as numeric and finite; scores are within 0–100; the wetness indicator is binary; wetness duration is non-negative; and stored categories match the documented score thresholds.

## Prediction-time availability

Air temperature can be read now. Current estimated leaf wetness can come from a current estimate or later sensor. Consecutive wetness hours require preceding hourly wetness history; one instantaneous reading is insufficient. This baseline estimates current environmental risk and uses no future observations. Forecasting and IoT integration are outside this stage.

## Risk categories

The reusable function applies: score 0 = No Risk; 0 < score < 25 = Low; 25 <= score < 75 = Moderate; 75 <= score <= 100 = High. The exact-zero No Risk boundary can be sensitive to imperfect regression predictions and should be reported during evaluation. Raw predictions should remain available so out-of-range outputs are visible.

## Limitations and later evaluation

The synthetic score generator is unavailable, and six repeated three-feature combinations have small score differences. Model performance will show imitation of synthetic labels, not confirmed field disease prediction. Historical NASA weather is kept separate for later evaluation; its environmental distribution differs from the scenarios and its proxy scores are not confirmed infections. Field measurements and confirmed disease outcomes are still needed for biological validation.

## Readiness

Preprocessing is ready for a preliminary Random Forest training stage using only the three approved inputs. Keep validation for model selection and reserve test for final evaluation. No model has been trained.
'''
(preprocessing_dir / 'Preprocessing_Report.md').write_text(report, encoding='utf-8')

source_hashes_after = {str(path.relative_to(ROOT)): sha256(path) for path in source_csvs}
assert source_hashes_after == source_hashes_before, 'A source CSV changed during preprocessing.'
print('Saved preprocessing report, config, and validation summary to:', preprocessing_dir)
print('All source CSV SHA-256 hashes are unchanged.')

Saved preprocessing report, config, and validation summary to: C:\Users\HP\Desktop\ML training\reports\black_sigatoka\preprocessing
All source CSV SHA-256 hashes are unchanged.
